# Module 02: Inference and Hypothesis Testing

## Intermediate Statistics for Econometrics

In this module, we transition from descriptive statistics to **statistical inference**—using sample data to make conclusions about populations. We'll cover the foundations: the normal distribution, standardization, the Central Limit Theorem, and hypothesis testing frameworks.

**Learning Objectives:**
- Understand the normal distribution and its properties
- Apply standardization to convert any normal variable to N(0,1)
- Grasp the Central Limit Theorem and its power
- Conduct and interpret hypothesis tests
- Distinguish Type I and Type II errors
- Know when to use z-tests vs. t-tests

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set random seed for reproducibility
np.random.seed(42)

# Configure plotting
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams['figure.figsize'] = (10, 6)

## 1. The Normal Distribution

The normal distribution is fundamental in statistics and econometrics. It is characterized by two parameters:
- **μ (mean)**: the center of the distribution
- **σ (standard deviation)**: controls the spread

### Probability Density Function (PDF)

$$f(x) = \frac{1}{\sigma\sqrt{2\pi}} e^{-\frac{(x-\mu)^2}{2\sigma^2}}$$

Key observations:
- The term $\frac{1}{\sigma\sqrt{2\pi}}$ is a normalizing constant ensuring the total area under the curve equals 1
- At x = μ, the height reaches its maximum: $f(\mu) = \frac{1}{\sigma\sqrt{2\pi}}$
- **Effect of σ**: As σ increases, the distribution becomes wider and shorter (maintaining total area = 1). As σ decreases, it becomes narrower and taller.

### Notation

We write $X \sim N(\mu, \sigma^2)$ to denote that X follows a normal distribution with mean μ and variance σ².

In [ ]:
# Visualize the normal distribution with different standard deviations
x = np.linspace(-4, 4, 1000)

fig, ax = plt.subplots(figsize=(10, 6))

# Plot three normal distributions with same mean (0) but different σ
for sigma in [0.5, 1.0, 2.0]:
    y = stats.norm.pdf(x, loc=0, scale=sigma)
    ax.plot(x, y, label=f'σ = {sigma}', linewidth=2)

ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('Density f(x)', fontsize=12)
ax.set_title('Normal Distribution: Effect of σ on Shape\n(μ = 0, Area under curve = 1)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Height at μ for different σ values:")
for sigma in [0.5, 1.0, 2.0]:
    height = 1 / (sigma * np.sqrt(2 * np.pi))
    print(f"  σ = {sigma}: f(μ) = 1/(σ√(2π)) = {height:.4f}")

## 2. Standardization

**Standardization** converts any normal variable into the **standard normal distribution** N(0, 1).

### The Z-Score

$$Z = \frac{X - \mu}{\sigma}$$

**Why is this useful?**
- One standard normal table/function (e.g., in Python) serves all normal distributions
- Z-scores measure distance from the mean in units of standard deviations
- A Z-score of 2 means "2 standard deviations above the mean"

### Properties
If $X \sim N(\mu, \sigma^2)$, then $Z = \frac{X - \mu}{\sigma} \sim N(0, 1)$

In [ ]:
# Example: Graduate salaries
# Suppose salary is distributed as N(μ=50000, σ=8000)
mu = 50000
sigma = 8000

# Specific salary values
salaries = np.array([50000, 54000, 42000, 58000])

# Compute Z-scores
z_scores = (salaries - mu) / sigma

df = pd.DataFrame({
    'Salary': salaries,
    'Z-score': z_scores
})
print("Standardization Example:")
print(f"Salary Distribution: N(μ={mu}, σ={sigma})\n")
print(df.to_string(index=False))
print(f"\nInterpretation:")
print(f"  - Salary $54,000: Z = 0.5 (half a std dev above mean)")
print(f"  - Salary $42,000: Z = -1.0 (one std dev below mean)")

In [ ]:
# Verify standardization transforms N(μ,σ²) to N(0,1)
# Generate data from N(50000, 8000²)
original_data = np.random.normal(loc=50000, scale=8000, size=10000)

# Standardize
standardized_data = (original_data - mu) / sigma

# Compare with N(0,1)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Original distribution
axes[0].hist(original_data, bins=50, density=True, alpha=0.7, label='Sample', color='skyblue')
x_orig = np.linspace(30000, 70000, 1000)
axes[0].plot(x_orig, stats.norm.pdf(x_orig, mu, sigma), 'r-', linewidth=2, label='N(50000, 8000²)')
axes[0].set_xlabel('Salary ($)', fontsize=11)
axes[0].set_ylabel('Density', fontsize=11)
axes[0].set_title('Original: N(50000, 8000²)', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Standardized distribution
axes[1].hist(standardized_data, bins=50, density=True, alpha=0.7, label='Sample', color='lightgreen')
x_std = np.linspace(-4, 4, 1000)
axes[1].plot(x_std, stats.norm.pdf(x_std, 0, 1), 'r-', linewidth=2, label='N(0, 1)')
axes[1].set_xlabel('Z-score', fontsize=11)
axes[1].set_ylabel('Density', fontsize=11)
axes[1].set_title('Standardized: N(0, 1)', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Original data: mean = {original_data.mean():.1f}, std = {original_data.std():.1f}")
print(f"Standardized data: mean = {standardized_data.mean():.6f}, std = {standardized_data.std():.6f}")

## 3. The Empirical Rule (68-95-99.7 Rule)

For any normal distribution:
- **68%** of observations fall within **μ ± 1σ**
- **95%** of observations fall within **μ ± 2σ**
- **99.7%** of observations fall within **μ ± 3σ**

This is a quick way to assess probabilities without computation. In econometrics, it helps identify outliers and assess data quality.

In [ ]:
# Worked example: Test scores
# Suppose test scores are N(75, 100), i.e., μ=75, σ=10

mu_score = 75
sigma_score = 10

# Calculate probabilities using scipy
prob_1sigma = stats.norm.cdf(mu_score + sigma_score, mu_score, sigma_score) - \
              stats.norm.cdf(mu_score - sigma_score, mu_score, sigma_score)
prob_2sigma = stats.norm.cdf(mu_score + 2*sigma_score, mu_score, sigma_score) - \
              stats.norm.cdf(mu_score - 2*sigma_score, mu_score, sigma_score)
prob_3sigma = stats.norm.cdf(mu_score + 3*sigma_score, mu_score, sigma_score) - \
              stats.norm.cdf(mu_score - 3*sigma_score, mu_score, sigma_score)

print("Test Scores: N(μ=75, σ=10)\n")
print(f"Range μ ± 1σ:  [65, 85]  → {prob_1sigma*100:.1f}% of students")
print(f"Range μ ± 2σ:  [55, 95]  → {prob_2sigma*100:.2f}% of students")
print(f"Range μ ± 3σ:  [45, 105] → {prob_3sigma*100:.3f}% of students")

# Visualize
x_scores = np.linspace(45, 105, 1000)
y_scores = stats.norm.pdf(x_scores, mu_score, sigma_score)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(x_scores, y_scores, 'b-', linewidth=2)

# Shade regions
x_1sigma = x_scores[(x_scores >= mu_score - sigma_score) & (x_scores <= mu_score + sigma_score)]
y_1sigma = stats.norm.pdf(x_1sigma, mu_score, sigma_score)
ax.fill_between(x_1sigma, y_1sigma, alpha=0.3, color='green', label='68% (±1σ)')

x_2sigma = x_scores[(x_scores >= mu_score - 2*sigma_score) & (x_scores <= mu_score + 2*sigma_score)]
y_2sigma = stats.norm.pdf(x_2sigma, mu_score, sigma_score)
ax.fill_between(x_2sigma, y_2sigma, alpha=0.15, color='blue', label='95% (±2σ)')

x_3sigma = x_scores[(x_scores >= mu_score - 3*sigma_score) & (x_scores <= mu_score + 3*sigma_score)]
y_3sigma = stats.norm.pdf(x_3sigma, mu_score, sigma_score)
ax.fill_between(x_3sigma, y_3sigma, alpha=0.05, color='red', label='99.7% (±3σ)')

ax.axvline(mu_score, color='black', linestyle='--', linewidth=1, alpha=0.5, label='μ')
ax.set_xlabel('Test Score', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Empirical Rule (68-95-99.7) for Test Scores', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. The Central Limit Theorem (CLT)

### Formal Statement

If $X_1, X_2, \ldots, X_n$ are independent and identically distributed (i.i.d.) random variables with mean μ and variance σ², then as n → ∞:

$$\bar{X} \approx N\left(\mu, \frac{\sigma^2}{n}\right)$$

Or equivalently, the standardized sample mean:

$$Z = \frac{\bar{X} - \mu}{\sigma/\sqrt{n}} \approx N(0, 1)$$

### Key Properties
- **E[X̄] = μ**: The sample mean is an unbiased estimator of the population mean
- **Var(X̄) = σ²/n**: The variance of the sample mean decreases with sample size
- **Standard Error** = $\sigma/\sqrt{n}$: Measures how precisely X̄ estimates μ

### The Power of CLT
The CLT is revolutionary: **the distribution of sample means is approximately normal, regardless of the shape of the original population distribution!** This is why normal inference works so broadly in statistics and econometrics.

In [ ]:
# Demonstration: CLT with a skewed distribution
# Start with an exponential distribution (highly skewed, not normal)

# Population parameters
lambda_param = 0.5
pop_mean = 1 / lambda_param  # 2
pop_std = 1 / lambda_param   # 2

# Sample parameters
n_per_sample = 50
n_samples = 10000

# Generate many samples and compute their means
sample_means = []
for _ in range(n_samples):
    sample = np.random.exponential(scale=1/lambda_param, size=n_per_sample)
    sample_means.append(np.mean(sample))

sample_means = np.array(sample_means)

# Theoretical properties under CLT
theoretical_mean = pop_mean
theoretical_se = pop_std / np.sqrt(n_per_sample)

print(f"Central Limit Theorem Demonstration")
print(f"={'='*50}")
print(f"Original Population: Exponential(λ={lambda_param})")
print(f"  Population mean: {pop_mean}")
print(f"  Population std: {pop_std}\n")
print(f"Sampling: n={n_per_sample} observations, {n_samples} samples\n")
print(f"Distribution of Sample Means:")
print(f"  Observed mean of means: {sample_means.mean():.4f}")
print(f"  Theoretical (E[X̄]): {theoretical_mean:.4f}")
print(f"  Observed std of means: {sample_means.std():.4f}")
print(f"  Theoretical (σ/√n = {theoretical_se:.4f})")

In [ ]:
# Visualize CLT: original distribution vs distribution of means
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Original population (exponential)
pop_sample = np.random.exponential(scale=1/lambda_param, size=10000)
axes[0].hist(pop_sample, bins=50, density=True, alpha=0.7, color='coral', edgecolor='black')
axes[0].set_xlabel('Value', fontsize=11)
axes[0].set_ylabel('Density', fontsize=11)
axes[0].set_title('Original Population\nExponential Distribution (Skewed)', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].text(0.95, 0.95, f'Mean = {pop_mean:.2f}\nStd = {pop_std:.2f}',
            transform=axes[0].transAxes, ha='right', va='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5),
            fontsize=10)

# Distribution of sample means
axes[1].hist(sample_means, bins=50, density=True, alpha=0.7, color='lightblue', edgecolor='black', label='Sample means')
x_normal = np.linspace(sample_means.min(), sample_means.max(), 1000)
axes[1].plot(x_normal, stats.norm.pdf(x_normal, theoretical_mean, theoretical_se),
            'r-', linewidth=2, label=f'N(μ, σ²/n)')
axes[1].set_xlabel('Sample Mean', fontsize=11)
axes[1].set_ylabel('Density', fontsize=11)
axes[1].set_title(f'Distribution of Sample Means (n={n_per_sample})\nApproximately Normal!', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].text(0.95, 0.95, f'Mean = {theoretical_mean:.2f}\nSE = {theoretical_se:.4f}',
            transform=axes[1].transAxes, ha='right', va='top',
            bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5),
            fontsize=10)

plt.tight_layout()
plt.show()

print("\nKey Insight: Even though the original population is highly skewed,")
print("the distribution of sample means is approximately NORMAL.")
print("This is the power of the Central Limit Theorem!")

## 5. From CLT to Inference

The CLT gives us a way to standardize the sample mean:

$$Z = \frac{\bar{X} - \mu_0}{\sigma/\sqrt{n}}$$

where:
- $\bar{X}$ is the sample mean
- $\mu_0$ is the hypothesized population mean
- $\sigma/\sqrt{n}$ is the standard error

Under the null hypothesis that the true mean is $\mu_0$, this Z-statistic follows N(0, 1) approximately. This is the foundation of hypothesis testing.

In [ ]:
# Worked example: Graduate salary inference
# Research question: Are graduate salaries significantly different from $50,000?

# Sample data
n = 100
x_bar = 52000  # sample mean
sigma = 10000  # population std (assumed known)
mu_0 = 50000   # hypothesized mean

# Compute test statistic
se = sigma / np.sqrt(n)
z_stat = (x_bar - mu_0) / se

print("Example: Testing Graduate Salaries")
print(f"{'='*50}")
print(f"Sample size: n = {n}")
print(f"Sample mean: X̄ = ${x_bar:,}")
print(f"Population std (known): σ = ${sigma:,}")
print(f"Hypothesized mean: μ₀ = ${mu_0:,}\n")
print(f"Standard Error: SE = σ/√n = {sigma}/√{n} = ${se:,.1f}\n")
print(f"Test Statistic: Z = (X̄ - μ₀)/SE")
print(f"             Z = ({x_bar} - {mu_0})/{se:,.1f}")
print(f"             Z = {z_stat:.4f}\n")
print(f"Interpretation: The sample mean is {z_stat:.2f} standard errors")
print(f"                above the hypothesized mean.")

In [ ]:
# Visualize the test statistic in context of N(0,1)
x_z = np.linspace(-4, 4, 1000)
y_z = stats.norm.pdf(x_z, 0, 1)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(x_z, y_z, 'b-', linewidth=2, label='N(0, 1)')

# Shade rejection region for two-tailed test at α=0.05
critical_value = 1.96
x_reject_right = x_z[x_z >= critical_value]
y_reject_right = stats.norm.pdf(x_reject_right, 0, 1)
ax.fill_between(x_reject_right, y_reject_right, alpha=0.3, color='red', label='Rejection region (α=0.05)')

x_reject_left = x_z[x_z <= -critical_value]
y_reject_left = stats.norm.pdf(x_reject_left, 0, 1)
ax.fill_between(x_reject_left, y_reject_left, alpha=0.3, color='red')

# Mark the observed test statistic
ax.axvline(z_stat, color='green', linestyle='--', linewidth=2.5, label=f'Observed Z = {z_stat:.2f}')
ax.axvline(critical_value, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label=f'Critical value = ±{critical_value}')
ax.axvline(-critical_value, color='red', linestyle='--', linewidth=1.5, alpha=0.7)

ax.set_xlabel('Z-statistic', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Hypothesis Test: Graduate Salaries (Two-Tailed, α=0.05)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nSince Z = {z_stat:.2f} > {critical_value} (critical value for α=0.05),")
print(f"we reject H₀. The evidence suggests graduate salaries differ from $50,000.")

## 6. Hypothesis Testing Framework

### Setup

A formal hypothesis test consists of:

1. **Null Hypothesis (H₀)**: The claim we're testing (e.g., μ = μ₀)
2. **Alternative Hypothesis (H₁)**: The complement to H₀
3. **Significance Level (α)**: The probability of rejecting H₀ when it's true (Type I error). Common: α = 0.05
4. **Test Statistic**: A function of the sample data (e.g., Z or t)
5. **Critical Value(s)**: The cutoff(s) that define the rejection region
6. **P-value**: The probability of observing data as extreme or more extreme than observed, given H₀ is true

### One-Tailed vs. Two-Tailed Tests

- **Two-tailed** (H₁: μ ≠ μ₀): Tests if the population mean differs in either direction. Critical region in both tails.
- **One-tailed** (H₁: μ > μ₀ or H₁: μ < μ₀): Tests if the mean is in a specific direction. Critical region in one tail.

### Decision Rule
- **Using p-value**: Reject H₀ if p-value < α
- **Using critical value**: Reject H₀ if |test statistic| > critical value (two-tailed)

In [ ]:
# Complete worked example: Graduate salaries hypothesis test
# H₀: μ = 25,000 (historical mean)
# H₁: μ ≠ 25,000 (two-tailed)
# α = 0.05

# Data
n_grad = 64
x_bar_grad = 26200
s_grad = 4800  # sample standard deviation (unknown σ, so use s)
mu_0_grad = 25000
alpha = 0.05

print("HYPOTHESIS TEST: Graduate Salaries")
print(f"{'='*60}")
print(f"\n1. SETUP")
print(f"   H₀: μ = ${mu_0_grad:,} (historical mean)")
print(f"   H₁: μ ≠ ${mu_0_grad:,} (two-tailed)")
print(f"   Significance level: α = {alpha}\n")
print(f"2. DATA")
print(f"   Sample size: n = {n_grad}")
print(f"   Sample mean: X̄ = ${x_bar_grad:,}")
print(f"   Sample std: s = ${s_grad:,}")
print(f"   (Note: σ unknown, so we use s as estimate)\n")

# Compute test statistic (t-statistic since σ is unknown)
se_grad = s_grad / np.sqrt(n_grad)
t_stat = (x_bar_grad - mu_0_grad) / se_grad
df = n_grad - 1  # degrees of freedom

print(f"3. TEST STATISTIC")
print(f"   SE = s/√n = {s_grad}/√{n_grad} = ${se_grad:,.1f}")
print(f"   t = (X̄ - μ₀)/SE = ({x_bar_grad} - {mu_0_grad})/{se_grad:,.1f}")
print(f"   t = {t_stat:.4f}")
print(f"   Degrees of freedom: df = n - 1 = {df}\n")

# Critical value for two-tailed t-test
t_crit = stats.t.ppf(1 - alpha/2, df)
print(f"4. CRITICAL VALUE")
print(f"   Two-tailed, α = {alpha}")
print(f"   Critical value: t_{df} = ±{t_crit:.4f}\n")

# P-value
p_value = 2 * (1 - stats.t.cdf(abs(t_stat), df))  # two-tailed
print(f"5. P-VALUE")
print(f"   p-value = {p_value:.6f}\n")

# Decision
reject = abs(t_stat) > t_crit or p_value < alpha
print(f"6. DECISION")
if reject:
    print(f"   Since |t| = {abs(t_stat):.4f} > {t_crit:.4f} (critical value)")
    print(f"   OR p-value = {p_value:.6f} < α = {alpha}")
    print(f"   → REJECT H₀ at the {alpha} significance level\n")
    print(f"   CONCLUSION: Graduate salaries differ significantly from $25,000.")
else:
    print(f"   Since |t| = {abs(t_stat):.4f} < {t_crit:.4f} (critical value)")
    print(f"   OR p-value = {p_value:.6f} > α = {alpha}")
    print(f"   → FAIL TO REJECT H₀ at the {alpha} significance level\n")
    print(f"   CONCLUSION: Insufficient evidence to conclude salaries differ.")

In [ ]:
# Visualize the t-test
x_t = np.linspace(-4, 4, 1000)
y_t = stats.t.pdf(x_t, df)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(x_t, y_t, 'b-', linewidth=2, label=f't-distribution (df={df})')

# Shade rejection regions
x_reject_r = x_t[x_t >= t_crit]
y_reject_r = stats.t.pdf(x_reject_r, df)
ax.fill_between(x_reject_r, y_reject_r, alpha=0.3, color='red', label='Rejection region')

x_reject_l = x_t[x_t <= -t_crit]
y_reject_l = stats.t.pdf(x_reject_l, df)
ax.fill_between(x_reject_l, y_reject_l, alpha=0.3, color='red')

# Mark critical value and observed statistic
ax.axvline(t_crit, color='red', linestyle='--', linewidth=1.5, alpha=0.7)
ax.axvline(-t_crit, color='red', linestyle='--', linewidth=1.5, alpha=0.7)
ax.axvline(t_stat, color='green', linestyle='--', linewidth=2.5, label=f'Observed t = {t_stat:.3f}')

ax.set_xlabel('t-statistic', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Hypothesis Test: Graduate Salaries (Two-Tailed, α=0.05)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Type I and Type II Errors

Hypothesis testing decisions can go wrong in two ways:

| Decision | H₀ True | H₀ False |
|----------|---------|----------|
| Reject H₀ | **Type I Error** (α) | Correct (Power = 1-β) |
| Fail to reject H₀ | Correct (1-α) | **Type II Error** (β) |

### Definitions

- **Type I Error (α)**: Reject H₀ when it's actually true. False positive.
  - Probability = α (significance level)
  - Controlled by the researcher

- **Type II Error (β)**: Fail to reject H₀ when it's actually false. False negative.
  - Probability = β
  - Decreases with larger sample size

- **Power (1-β)**: Probability of correctly rejecting H₀ when it's false.
  - Higher power is desirable
  - Increases with sample size

### The α-β Trade-off

Decreasing α (being more conservative, harder to reject H₀) increases β. There's a trade-off: lowering the Type I error rate often increases the Type II error rate.

In [ ]:
# Visualize Type I and Type II errors
# Scenario: Testing H₀: μ = μ₀ vs H₁: μ = μ₁

mu_0 = 0  # null hypothesis mean
mu_1 = 2.5  # alternative hypothesis mean
sigma = 1
n = 100
alpha = 0.05

# Under H₀, sample mean ~ N(μ₀, σ²/n)
se = sigma / np.sqrt(n)

# Critical value for two-tailed test
z_crit = stats.norm.ppf(1 - alpha/2)

x_range = np.linspace(-1, 4, 1000)

# Distribution under H₀
y_h0 = stats.norm.pdf(x_range, mu_0, se)

# Distribution under H₁
y_h1 = stats.norm.pdf(x_range, mu_1, se)

fig, ax = plt.subplots(figsize=(12, 6))

# Plot distributions
ax.plot(x_range, y_h0, 'b-', linewidth=2.5, label='Under H₀: μ = 0')
ax.plot(x_range, y_h1, 'r-', linewidth=2.5, label='Under H₁: μ = 2.5')

# Type I error region (reject H₀ when it's true)
x_alpha_right = x_range[x_range >= mu_0 + z_crit * se]
y_alpha_right = stats.norm.pdf(x_alpha_right, mu_0, se)
ax.fill_between(x_alpha_right, y_alpha_right, alpha=0.4, color='blue', label='Type I Error (α)')

x_alpha_left = x_range[x_range <= mu_0 - z_crit * se]
y_alpha_left = stats.norm.pdf(x_alpha_left, mu_0, se)
ax.fill_between(x_alpha_left, y_alpha_left, alpha=0.4, color='blue')

# Type II error region (fail to reject H₀ when it's false)
x_beta = x_range[(x_range > mu_0 - z_crit * se) & (x_range < mu_0 + z_crit * se)]
y_beta = stats.norm.pdf(x_beta, mu_1, se)
ax.fill_between(x_beta, y_beta, alpha=0.4, color='red', label='Type II Error (β)')

# Critical values
ax.axvline(mu_0 + z_crit * se, color='black', linestyle='--', linewidth=1.5, alpha=0.6)
ax.axvline(mu_0 - z_crit * se, color='black', linestyle='--', linewidth=1.5, alpha=0.6)

ax.set_xlabel('Sample Mean', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Type I and Type II Errors', fontsize=13, fontweight='bold')
ax.legend(fontsize=11, loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Calculate actual β and power
cutoff_upper = mu_0 + z_crit * se
cutoff_lower = mu_0 - z_crit * se
beta = stats.norm.cdf(cutoff_upper, mu_1, se) - stats.norm.cdf(cutoff_lower, mu_1, se)
power = 1 - beta

print(f"Error Analysis")
print(f"{'='*50}")
print(f"Under H₀: μ = {mu_0}, σ = {sigma}, n = {n}")
print(f"Under H₁: μ = {mu_1}, σ = {sigma}, n = {n}\n")
print(f"Significance level: α = {alpha:.4f}")
print(f"Type II error: β = {beta:.4f}")
print(f"Power: 1 - β = {power:.4f}\n")
print(f"Interpretation:")
print(f"  - {alpha*100:.1f}% chance of rejecting H₀ when it's true (Type I)")
print(f"  - {beta*100:.1f}% chance of failing to reject H₀ when μ=2.5 (Type II)")
print(f"  - {power*100:.1f}% chance of correctly rejecting H₀ when μ=2.5 (Power)")

## 8. Statistical vs. Economic Significance

### The Distinction

- **Statistical Significance**: The result is unlikely due to chance (p < α). Depends heavily on sample size.
- **Economic Significance**: The effect is large enough to matter in practice. Depends on the effect size and context.

### Why They Differ

With a very large sample size, even a tiny difference from the null hypothesis becomes statistically significant—but may be economically irrelevant.

**Example**: A salary difference of $1 with n=1,000,000 might be statistically significant (p < 0.05) but economically meaningless.

### In Econometrics

Always report effect sizes (e.g., regression coefficients, elasticities, percentage changes) alongside p-values. A 1% wage increase is very different from a 50% wage increase, even if both are statistically significant.

In [ ]:
# Demonstrate: statistical significance without economic significance
# Small effect with large sample → statistically significant but economically trivial

# Scenario 1: Small effect, large sample
mu_test = 25000.10  # Only $0.10 difference!
n_large = 10000
sigma_salary = 5000

# Sample mean under alternative
x_bar_1 = mu_test
se_1 = sigma_salary / np.sqrt(n_large)
z_stat_1 = (x_bar_1 - 25000) / se_1
p_val_1 = 2 * (1 - stats.norm.cdf(abs(z_stat_1)))

print("Statistical vs. Economic Significance")
print(f"{'='*60}")
print(f"\nScenario 1: Small effect + large sample")
print(f"  H₀: μ = $25,000")
print(f"  Sample mean: $25,000.10 (only 10 cents different!)")
print(f"  Sample size: n = {n_large:,}")
print(f"  Z-statistic: {z_stat_1:.4f}")
print(f"  p-value: {p_val_1:.10f}")
print(f"  → Statistically significant (p < 0.05): {p_val_1 < 0.05}")
print(f"  → Economically significant? NO! Effect is negligible.")

# Scenario 2: Large effect, small sample
mu_test_2 = 27000  # $2,000 difference
n_small = 25

x_bar_2 = mu_test_2
se_2 = sigma_salary / np.sqrt(n_small)
z_stat_2 = (x_bar_2 - 25000) / se_2
p_val_2 = 2 * (1 - stats.norm.cdf(abs(z_stat_2)))

print(f"\nScenario 2: Large effect + small sample")
print(f"  H₀: μ = $25,000")
print(f"  Sample mean: $27,000 ($2,000 increase)")
print(f"  Sample size: n = {n_small}")
print(f"  Z-statistic: {z_stat_2:.4f}")
print(f"  p-value: {p_val_2:.6f}")
print(f"  → Statistically significant (p < 0.05): {p_val_2 < 0.05}")
print(f"  → Economically significant? YES! 8% wage increase.")

print(f"\n{'='*60}")
print(f"Key Lesson:")
print(f"  Always report effect sizes, not just p-values.")
print(f"  A significant p-value tells you something is likely NOT zero.")
print(f"  The coefficient/effect tells you how MUCH it matters.")

## 9. Z-Test vs. t-Test

### When to Use Which?

| Condition | Test | Comment |
|-----------|------|----------|
| σ **known**, large n | Z-test | Use standard normal N(0,1) |
| σ **unknown**, small n | t-test | Use t-distribution with df=n-1 |
| σ **unknown**, large n (n>30) | Either | t-test is technically correct but Z-test is close |

### The t-Distribution

When we estimate σ from the sample (using s), we introduce additional uncertainty. The t-distribution accounts for this:
- Fatter tails than normal distribution (more uncertainty)
- Controlled by degrees of freedom (df = n - 1)
- As n → ∞, t-distribution → N(0,1)

### Test Statistic

**Z-test** (σ known):
$$Z = \frac{\bar{X} - \mu_0}{\sigma/\sqrt{n}} \sim N(0,1)$$

**t-test** (σ unknown):
$$t = \frac{\bar{X} - \mu_0}{s/\sqrt{n}} \sim t_{n-1}$$

In [ ]:
# Compare t-distribution to normal distribution
x = np.linspace(-4, 4, 1000)

fig, ax = plt.subplots(figsize=(10, 6))

# Standard normal
ax.plot(x, stats.norm.pdf(x, 0, 1), 'k-', linewidth=2.5, label='N(0,1)')

# t-distributions with different df
for df_val in [1, 5, 10, 30]:
    ax.plot(x, stats.t.pdf(x, df_val), '--', linewidth=2, label=f't(df={df_val})')

ax.set_xlabel('Test Statistic', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('t-Distribution vs. Normal Distribution\n(t becomes more normal as df increases)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 0.45)
plt.tight_layout()
plt.show()

print("Key observations:")
print("  - t-distribution has fatter tails (more uncertainty)")
print("  - With df=5 (n=6), critical values are ~2.6 vs ~2.0 for normal")
print("  - As df increases, t → N(0,1)")
print("  - For practical purposes, they're close when df > 30")

In [ ]:
# Example: Comparing Z-test (σ known) vs t-test (σ unknown)

# Sample data (hypothetical)
data = np.array([52000, 54000, 48000, 56000, 50000, 53000, 49000, 51000])  # n=8
n = len(data)
x_bar = np.mean(data)
s = np.std(data, ddof=1)  # sample std (unbiased)
mu_0 = 50000

print("Example: Salary Inference (n=8 small sample)")
print(f"{'='*60}")
print(f"\nData: {data}")
print(f"Sample mean: X̄ = ${x_bar:,.0f}")
print(f"Sample std: s = ${s:,.0f}")
print(f"H₀: μ = ${mu_0:,}\n")

# Scenario A: σ known (assume σ = population std estimate)
sigma_known = s  # Assume we know this
se_z = sigma_known / np.sqrt(n)
z_stat = (x_bar - mu_0) / se_z
p_val_z = 2 * (1 - stats.norm.cdf(abs(z_stat)))
z_crit = stats.norm.ppf(1 - 0.05/2)

print(f"SCENARIO A: σ KNOWN = ${sigma_known:,.0f}")
print(f"  SE = σ/√n = {sigma_known:,.0f}/√{n} = ${se_z:,.1f}")
print(f"  Z = (X̄ - μ₀)/SE = {z_stat:.4f}")
print(f"  Critical value (α=0.05, two-tailed): ±{z_crit:.4f}")
print(f"  p-value = {p_val_z:.6f}")
print(f"  Decision: {'Reject H₀' if abs(z_stat) > z_crit else 'Fail to reject H₀'}\n")

# Scenario B: σ unknown (use t-test)
df = n - 1
se_t = s / np.sqrt(n)
t_stat = (x_bar - mu_0) / se_t
p_val_t = 2 * (1 - stats.t.cdf(abs(t_stat), df))
t_crit = stats.t.ppf(1 - 0.05/2, df)

print(f"SCENARIO B: σ UNKNOWN (use t-test)")
print(f"  SE = s/√n = {s:,.0f}/√{n} = ${se_t:,.1f}")
print(f"  t = (X̄ - μ₀)/SE = {t_stat:.4f}")
print(f"  Degrees of freedom: df = n - 1 = {df}")
print(f"  Critical value (α=0.05, two-tailed, df={df}): ±{t_crit:.4f}")
print(f"  p-value = {p_val_t:.6f}")
print(f"  Decision: {'Reject H₀' if abs(t_stat) > t_crit else 'Fail to reject H₀'}\n")

print(f"Comparison:")
print(f"  Z-critical ({z_crit:.4f}) < t-critical ({t_crit:.4f})")
print(f"  → t-test is more conservative (harder to reject H₀)")
print(f"  This accounts for uncertainty in estimating σ.")

## 10. Exercises

### Exercise 1: Standardization
Suppose household income is distributed as N(μ=60,000, σ=15,000). What is the Z-score for a household with income $75,000? What proportion of households earn more than $75,000?

**Hint**: Use `stats.norm.sf()` (survival function = 1 - CDF) or `1 - stats.norm.cdf()`

In [ ]:
# Exercise 1: Standardization
# YOUR CODE HERE

### Exercise 2: Empirical Rule
A firm's production output per worker is distributed as N(μ=100 units/day, σ=20 units/day). Using the empirical rule, estimate what percentage of workers produce between 60 and 140 units per day. Verify your answer using scipy.stats.

In [ ]:
# Exercise 2: Empirical Rule
# YOUR CODE HERE

### Exercise 3: Central Limit Theorem
Simulate the Central Limit Theorem with a **uniform distribution** U(0, 10). Draw 5,000 samples of size n=40 from this distribution, compute the sample mean for each, and visualize the distribution of these sample means. Does it look approximately normal?

In [ ]:
# Exercise 3: Central Limit Theorem with Uniform Distribution
# YOUR CODE HERE

### Exercise 4: Hypothesis Test (One-Tailed)
A coffee shop claims their average cup contains 12 oz of coffee. A consumer advocacy group collects a sample of 36 cups and finds X̄ = 11.8 oz, s = 0.4 oz. Test H₀: μ = 12 vs H₁: μ < 12 at α = 0.05. Use a t-test. Do you reject the null hypothesis?

In [ ]:
# Exercise 4: One-Tailed Hypothesis Test
# YOUR CODE HERE

### Exercise 5: Type I and Type II Errors (Conceptual)
A bank is testing whether a new loan approval algorithm increases mean loan approval time.

a) **Define Type I Error**: What would it mean in this context? What's the consequence of approving the algorithm when it doesn't actually help?

b) **Define Type II Error**: What would it mean here? What's the consequence of rejecting the algorithm when it would actually help?

c) **Which is worse?** Discuss the trade-offs from the bank's perspective.

In [ ]:
# Exercise 5: Type I and Type II Errors (Conceptual)
# Write your answer below as comments or markdown

# YOUR ANSWER HERE

## Key Takeaways

1. **Normal Distribution**: Characterized by μ and σ. Standard normal N(0,1) is universal via standardization (Z-score).

2. **Standardization**: Z = (X - μ)/σ converts any normal to N(0,1). Z-scores measure distance in standard deviations.

3. **Central Limit Theorem**: Sample means are approximately normal regardless of population shape. This is the foundation of inference.

4. **Standard Error**: SE = σ/√n. Precision increases with sample size. This is why large samples matter.

5. **Hypothesis Testing**: Formal framework with H₀, H₁, α, test statistic, critical value, and p-value.

6. **Type I vs. II Errors**: There's a trade-off. α is controlled; β depends on sample size and effect size. Power = 1 - β.

7. **Statistical ≠ Economic Significance**: Large samples make small effects significant. Always report effect sizes.

8. **Z-test vs. t-test**: Use Z when σ is known (rare). Use t when σ is unknown and especially for small samples.

9. **t-distribution**: Fatter tails than normal; accounts for estimating σ. Converges to normal as n increases.

10. **Econometrics Application**: These tools enable us to test causal claims, compare groups, and assess policy impacts with quantified uncertainty.

---

## Next Module: Confidence Intervals

In Module 03, we'll flip the perspective: instead of testing claims about μ, we'll **estimate** μ using confidence intervals. We'll build on the normal and t-distributions to construct intervals like CI = X̄ ± t* × SE, and learn when and how to interpret them correctly in econometric applications.

---

**Footer**

This notebook is part of the open course *'Intermediate Statistics for Econometrics'* by Andrea Sarcina.  
Published at [cinosar.github.io](https://cinosar.github.io) — Licensed under CC BY 4.0